# Fish detector: pretrained Community Fish Detector RF-DETR Medium (no training) - Colab, ~15 min

Highest-scoring pretrained fish detector found: **CFD RF-DETR Medium 1024** (AP .609 on the CFD validation set;
trained on 1.9 M images / 935 k fish boxes from 17 open datasets; Apache-2.0;
github.com/filippovarini/community-fish-detector). Used as-is, like the pretrained coral SegFormer.

TrashCan is **not** in its training data, so the TrashCan test is a clean comparison with the current detector
(TrashCan test, conf 0.18, IoU 0.5: P 0.407, R 0.159, F1 0.229).

- Confidence threshold chosen on **TrashCan val** (max F1), applied once to **TrashCan test**.
- Same matching rule as all project evaluations (greedy, highest confidence first, IoU >= 0.5).
- 95% intervals: bootstrap over the 53 test videos. Also AP50 and image-level "fish present" accuracy.

**Runtime:** T4 GPU. **Needs:** `MyDrive/fish/fish_dataset.zip`. **Bring back:** `MyDrive/fish/fish_cfd_rfdetr_results.json`.

In [ ]:
!nvidia-smi -L
!pip install -q rfdetr supervision
!wget -q -nc -O /content/cfd_rfdetr_medium.pth https://github.com/filippovarini/community-fish-detector/releases/download/2026.07.06-release/cfd-rf-detr-medium-1024-2026.03.24.cp-011.20260706-release.pth

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, re, json, zipfile, collections
import numpy as np
from pathlib import Path
from PIL import Image
DRIVE_DIR = '/content/drive/MyDrive/fish'
ZIP = f'{DRIVE_DIR}/fish_dataset.zip'
assert os.path.exists(ZIP), f'Upload fish_dataset.zip to {ZIP}'
if not os.path.exists('/content/fish/processed_fish'):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content/fish')
V1 = Path('/content/fish/processed_fish')
print({s: len(os.listdir(V1 / s / 'images')) for s in ('val', 'test')})

In [ ]:
from rfdetr import from_checkpoint
model = from_checkpoint('/content/cfd_rfdetr_medium.pth')

def gt_boxes(split, name):
    w, h = Image.open(V1 / split / 'images' / name).size
    boxes = []
    for line in (V1 / split / 'labels' / (Path(name).stem + '.txt')).read_text().splitlines():
        v = [float(x) for x in line.split()[1:]]
        if len(v) == 4:
            xc, yc, bw, bh = v
            boxes.append([(xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h])
        elif len(v) >= 6:   # polygon -> box
            xs, ys = np.clip(v[0::2], 0, 1), np.clip(v[1::2], 0, 1)
            boxes.append([xs.min() * w, ys.min() * h, xs.max() * w, ys.max() * h])
    return np.array(boxes, np.float32).reshape(-1, 4)

def predict(split, names):
    preds = {}
    for i, n in enumerate(names):
        d = model.predict(Image.open(V1 / split / 'images' / n).convert('RGB'), threshold=0.02)
        preds[n] = (np.asarray(d.xyxy, np.float32).reshape(-1, 4), np.asarray(d.confidence, np.float32).reshape(-1))
        if i % 200 == 0:
            print(split, i, '/', len(names))
    return preds

def iou(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def match_flags(gt, boxes, confs):
    """Greedy IoU>=0.5 matching, highest confidence first (project rule). Returns (conf, is_tp) per detection."""
    order = np.argsort(-confs)
    ious = iou(boxes, gt) if len(boxes) and len(gt) else np.zeros((len(boxes), len(gt)))
    used, out = set(), []
    for i in order:
        best_g, best_iou = -1, 0.5
        for g in range(len(gt)):
            if g not in used and ious[i, g] >= best_iou:
                best_g, best_iou = g, ious[i, g]
        if best_g >= 0:
            used.add(best_g)
        out.append((float(confs[i]), best_g >= 0))
    return out

def counts(gt, boxes, confs, conf):
    keep = confs >= conf
    flags = match_flags(gt, boxes[keep], confs[keep])
    tp = sum(f for _, f in flags)
    return tp, len(flags) - tp, len(gt) - tp

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)

def load(split):
    names = sorted(os.listdir(V1 / split / 'images'))
    return names, {n: gt_boxes(split, n) for n in names}, predict(split, names)

val = load('val'); test = load('test')

In [ ]:
def totals(data, conf):
    names, gts, preds = data
    per = {n: counts(gts[n], *preds[n], conf) for n in names}
    return per, tuple(sum(v[k] for v in per.values()) for k in range(3))

table = [(prf(*totals(val, c)[1])[2], float(c)) for c in np.round(np.arange(0.05, 0.91, 0.05), 2)]
val_f1, conf = max(table)
print('val F1 by threshold:', [(c, round(f, 3)) for f, c in table])
print('chosen on val:', conf, 'val F1', round(val_f1, 3))

names, gts, preds = test
per, (tp, fp, fn) = totals(test, conf)
p, r, f1 = prf(tp, fp, fn)
vid = lambda n: re.match(r'vid_(\d+)_frame', n).group(1)
vids = collections.defaultdict(list)
for n in names:
    vids[vid(n)].append(n)
rng = np.random.default_rng(0); keys = list(vids); boot = []
for _ in range(1000):
    s = [n for k in rng.choice(keys, len(keys)) for n in vids[k]]
    boot.append(prf(*(sum(per[n][k] for n in s) for k in range(3))))
ci = lambda j: [round(float(np.percentile([b[j] for b in boot], q)), 3) for q in (2.5, 97.5)]

# AP50 (all-points), all detections >= 0.02
flags = [f for n in names for f in match_flags(gts[n], *preds[n])]
flags.sort(key=lambda t: -t[0]); n_gt = sum(len(gts[n]) for n in names)
tps = np.cumsum([f for _, f in flags]); fps = np.cumsum([not f for _, f in flags])
rec = tps / max(n_gt, 1); prec = tps / np.maximum(tps + fps, 1)
mrec = np.concatenate([[0], rec, [1]]); mpre = np.concatenate([[1], prec, [0]])
mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
ap50 = float(np.sum((mrec[1:] - mrec[:-1]) * mpre[1:]))

y = np.array([len(gts[n]) > 0 for n in names]); yp = np.array([bool((preds[n][1] >= conf).any()) for n in names])
itp = int((y & yp).sum())
RESULTS = {
    'model': 'Community Fish Detector RF-DETR Medium 1024 (cfd-2026.05.13), pretrained, no fine-tuning',
    'test_split': 'TrashCan test (53 videos; not in CFD training data)', 'matching': 'greedy IoU>=0.5',
    'conf_chosen_on_val': conf, 'val_f1_at_conf': round(val_f1, 3),
    'test': {'precision': round(p, 3), 'recall': round(r, 3), 'f1': round(f1, 3), 'tp': int(tp), 'fp': int(fp), 'fn': int(fn),
             'precision_ci': ci(0), 'recall_ci': ci(1), 'f1_ci': ci(2), 'ap50': round(ap50, 3), 'n_fish': int(n_gt)},
    'image_level_fish_present': {'accuracy': round(float((y == yp).mean()), 3), 'precision': round(itp / max(int(yp.sum()), 1), 3),
        'recall': round(itp / max(int(y.sum()), 1), 3), 'n_positive': int(y.sum()), 'n_images': len(names),
        'always_majority_baseline': round(float(max(y.mean(), 1 - y.mean())), 3)},
    'baseline_trashcan_test': {'model': 'old 16-class detector, conf 0.18', 'precision': 0.407, 'recall': 0.159, 'f1': 0.229},
}
json.dump(RESULTS, open(f'{DRIVE_DIR}/fish_cfd_rfdetr_results.json', 'w'), indent=2)
print(json.dumps(RESULTS, indent=1))
print(f'\nTrashCan TEST F1 {f1:.3f} {ci(2)} vs baseline 0.229 | AP50 {ap50:.3f}')